# Auslan-Daily → SMPL-X pilot (200 Communication clips) for SignSparK

Converts a pilot set of Auslan-Daily Communication clips into SignSparK's LMDB format and checks the fit quality **before** converting all 14k clips.

Pipeline per clip:
1. **Body**: NLF (NeurIPS 2024) fits SMPL-X on each frame (whole frame as the person box, fixed known camera intrinsics).
2. **Hands**: WiLoR (CVPR 2025) on hand boxes built from our existing rtmlib keypoints, so left/right comes from the already-verified 2D tracking.
3. **Assembly** (`smplx_pilot.py`, unit-tested locally, 25/25 against SignSparK's own loader code): WiLoR fingers; wrist = WiLoR hand orientation composed with NLF's elbow; left hand stored in SMPL-X-left convention as SignSparK expects; gaps interpolated; light temporal smoothing. Face = NLF jaw + zero expression (no face model in the pilot, so mouthing is lost).
4. **Segments**: rough sign segmentation from rtmlib wrist height/speed (stand-in for FAST, which is embargoed until end of Sept 2026); SignSparK only uses it to pick keyframes.
5. **LMDB** with language tag `Auslan`.

Quality gates (section 8–9): NLF self-consistency; 2D reprojection of arms and hands against rtmlib, **assembled vs NLF-only hands**; hand coverage; overlay videos; SignSparK's loader reads the LMDB.

**Before running**, on Drive:
- `MyDrive/smplx_models/smplx/SMPLX_NEUTRAL.npz` (already there from the zero-shot notebook);
- `MyDrive/mano_models/MANO_RIGHT.pkl`: register at https://mano.is.tue.mpg.de/, download "Models & Code", take `mano_v1_2/models/MANO_RIGHT.pkl`.

Licences: NLF and WiLoR weights, MANO and SMPL-X are non-commercial research only.
Outputs go to `MyDrive/auslan_work/smplx_pilot/`; per-clip raw fits are saved, so a disconnect only loses the clip in progress.

## 1. Runtime, Drive and inputs

In [1]:
import os, sys, glob, json, io, shutil, subprocess, tarfile, time, zipfile
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'NO GPU')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
OUT = f'{WORK}/smplx_pilot'
for d in ('params', 'lmdb/train', 'videos'):
    os.makedirs(f'{OUT}/{d}', exist_ok=True)

MANIFEST = f'{WORK}/manifest.jsonl'
EXCLUDE = f'{WORK}/excluded.txt'
SIGNER_ZIP = f'{DRIVE}/Auslan-Daily/Auslan-Daily Communication/Signer Only Video Clip/Signer.zip'
SMPLX_NPZ = f'{DRIVE}/smplx_models/smplx/SMPLX_NEUTRAL.npz'
MANO_PKL = f'{DRIVE}/mano_models/MANO_RIGHT.pkl'
missing = [p for p in (MANIFEST, EXCLUDE, SIGNER_ZIP, SMPLX_NPZ, MANO_PKL) if not os.path.exists(p)]
if not glob.glob(f'{WORK}/pose/chunk_*.tar'):
    missing.append(f'{WORK}/pose/chunk_*.tar')
if missing:
    raise SystemExit('missing on Drive:\n  ' + '\n  '.join(missing))
print('inputs OK')

Mounted at /content/drive
inputs OK


## 2. Dependencies, models and the pilot module

- WiLoR (pinned commit) and its weights; MANO is converted from `.pkl` to `.npz` using the chumpy source on `sys.path` (chumpy's `setup.py` does not install on current pip, but its code imports fine on numpy 2), and WiLoR's config is pointed at the `.npz`.
- NLF v0.3.2 TorchScript (GitHub release).
- SignSparK (pinned) is cloned only for the loader check in section 10.

In [2]:
!pip -q install smplx==0.1.28 pytorch-lightning yacs timm einops scikit-image pyrender trimesh lmdb
WILOR = '/content/WiLoR'
WILOR_COMMIT = 'fcb911312a38fa8badd30d9656a167485d61b8f9'
if not os.path.isdir(f'{WILOR}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/rolpotamias/WiLoR.git', WILOR], check=True)
subprocess.run(['git', '-C', WILOR, 'checkout', '-q', WILOR_COMMIT], check=True)
ckpt = f'{WILOR}/pretrained_models/wilor_final.ckpt'
if not os.path.exists(ckpt):
    subprocess.run(['wget', '-q', '-O', ckpt, 'https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/wilor_final.ckpt'], check=True)
print('WiLoR ckpt', round(os.path.getsize(ckpt) / 1e6), 'MB')

NLF_PATH = '/content/nlf_l_multi_0.3.2.torchscript'
if not os.path.exists(NLF_PATH):
    subprocess.run(['wget', '-q', '-O', NLF_PATH, 'https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'], check=True)
print('NLF', round(os.path.getsize(NLF_PATH) / 1e6), 'MB')

# MANO_RIGHT.pkl stores some arrays as chumpy operations; only real chumpy can evaluate them.
# The current source imports on numpy 2 without installing (its setup.py does not build on new pip).
CHUMPY = '/content/chumpy'
if not os.path.isdir(f'{CHUMPY}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mattloper/chumpy.git', CHUMPY], check=True)
subprocess.run(['git', '-C', CHUMPY, 'checkout', '-q', '580566eafc9ac68b2614b64d6f7aaa84eebb70da'], check=True)

SSK = '/content/SignSparK'
if not os.path.isdir(f'{SSK}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/JianHe0628/SignSparK.git', SSK], check=True)
subprocess.run(['git', '-C', SSK, 'checkout', '-q', '7b9b48360c4f5c26ceb0ff5a8d88424ade4e586d'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.0 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.0/750.0 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.0 MB/s eta 0:00:00
WiLoR ckpt 2565 MB
NLF 493 MB


CompletedProcess(args=['git', '-C', '/content/SignSparK', 'checkout', '-q', '7b9b48360c4f5c26ceb0ff5a8d88424ade4e586d'], returncode=0)

The pilot module (same file as `auslan_smplx/smplx_pilot.py` in the project, tested by `auslan_smplx/test_smplx_pilot.py`).

In [ ]:
%%writefile /content/smplx_pilot.py
"""Auslan-Daily clip -> SMPL-X rotations -> SignSparK LMDB record.

Body comes from NLF (SMPL-X fit on the whole frame), hands from WiLoR (MANO,
run on hand boxes taken from our rtmlib keypoints, so handedness comes from the
already-verified 2D tracking rather than WiLoR's own detector). This module is
pure numpy/scipy so it can be tested without either model.

Conventions (checked against the SignSparK and WiLoR code, not assumed):
  * rotation 6D = first two ROWS of the matrix (SignSparK `_matrix_to_rot6d_np`);
  * WiLoR's MANO is a `smplx.MANOLayer`, which adds no mean hand pose, so its
    hand_pose is absolute and matches SignSparK's `flat_hand_mean=True`;
  * WiLoR predicts a left hand on the mirrored crop, i.e. as a right hand. That
    raw output is SignSparK's "right-hand (WiLoR) convention". The LMDB stores
    the left hand in true SMPL-X-left convention, F @ R @ F with
    F = diag(1, -1, -1), and SignSparK's loader applies the same conjugation to
    get back to the raw WiLoR form (`flip_left_hand=True`);
  * the same raw left-hand global orientation, mirrored into the real camera,
    is M @ R @ M with M = diag(-1, 1, 1) (conjugating by M or by F is the same
    map, since F = -M);
  * body_features holds the 21 SMPL-X body joints (1..21) as local 6D; the
    SignSparK loader keeps joints 12..21 (neck, collars, head, shoulders,
    elbows, wrists). Wrist rotations come from WiLoR's hand orientation
    composed with NLF's elbow: R_wrist_local = G_elbow^T @ R_hand_camera;
  * face_features = jaw 6D (NLF) + 50 expression coefficients (zeros here: no
    face model in the pilot).
"""

from __future__ import annotations

import io
import pickle

import numpy as np
from scipy.signal import savgol_filter
from scipy.spatial.transform import Rotation

# SMPL-X kinematic tree for the 55 joints (smplx package order).
SMPLX_PARENTS = np.array([-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 15, 15, 15,
                          20, 25, 26, 20, 28, 29, 20, 31, 32, 20, 34, 35, 20, 37, 38,
                          21, 40, 41, 21, 43, 44, 21, 46, 47, 21, 49, 50, 21, 52, 53])
L_ELBOW, R_ELBOW, L_WRIST, R_WRIST, JAW = 18, 19, 20, 21, 22
L_HAND = slice(25, 40)
R_HAND = slice(40, 55)
F_FLIP = np.diag([1.0, -1.0, -1.0])     # SignSparK left-hand flip
M_MIRROR = np.diag([-1.0, 1.0, 1.0])    # image mirror about x

# COCO-WholeBody (rtmlib) indices.
COCO_L_SHOULDER, COCO_R_SHOULDER, COCO_L_WRIST, COCO_R_WRIST = 5, 6, 9, 10
COCO_L_HAND = slice(91, 112)
COCO_R_HAND = slice(112, 133)
# COCO-WholeBody / OpenPose hand order: wrist, thumb 1-4, index 1-4, middle, ring, pinky.
# SMPL-X per hand: index1-3, middle1-3, pinky1-3, ring1-3, thumb1-3 (+15 for the right hand),
# fingertips 66-70 (left thumb, index, middle, ring, pinky) and 71-75 (right).
_OP_FROM_SMPLX_LEFT = [20, 37, 38, 39, 66, 25, 26, 27, 67, 28, 29, 30, 68, 34, 35, 36, 69, 31, 32, 33, 70]
OP_FROM_SMPLX = {'left': _OP_FROM_SMPLX_LEFT,
                 'right': [21] + [j + 15 if 25 <= j < 40 else j + 5 for j in _OP_FROM_SMPLX_LEFT[1:]]}
# COCO body joint -> SMPL-X joint, for the arm checks.
COCO_TO_SMPLX_ARMS = {5: 16, 6: 17, 7: 18, 8: 19, 9: 20, 10: 21}


# --------------------------------------------------------------------------- rotations
def aa_to_mat(aa: np.ndarray) -> np.ndarray:
    aa = np.asarray(aa, np.float64)
    return Rotation.from_rotvec(aa.reshape(-1, 3)).as_matrix().reshape(*aa.shape[:-1], 3, 3)


def mat_to_aa(R: np.ndarray) -> np.ndarray:
    R = np.asarray(R, np.float64)
    return Rotation.from_matrix(R.reshape(-1, 3, 3)).as_rotvec().reshape(*R.shape[:-2], 3)


def mat_to_6d(R: np.ndarray) -> np.ndarray:
    R = np.asarray(R)
    return R[..., :2, :].reshape(*R.shape[:-2], 6)


def sixd_to_mat(d6: np.ndarray) -> np.ndarray:
    """Gram-Schmidt on rows, identical to SignSparK `_rot6d_to_matrix_np`."""
    d6 = np.asarray(d6, np.float64)
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = a1 / np.linalg.norm(a1, axis=-1, keepdims=True)
    b2 = a2 - np.sum(b1 * a2, axis=-1, keepdims=True) * b1
    b2 = b2 / np.linalg.norm(b2, axis=-1, keepdims=True)
    return np.stack((b1, b2, np.cross(b1, b2)), axis=-2)


def global_rotations(local: np.ndarray, upto: int = 22) -> np.ndarray:
    """(T, 55, 3, 3) local -> (T, upto, 3, 3) global, via SMPLX_PARENTS."""
    G = np.empty_like(local[:, :upto])
    for j in range(upto):
        p = SMPLX_PARENTS[j]
        G[:, j] = local[:, j] if p < 0 else G[:, p] @ local[:, j]
    return G


# --------------------------------------------------------------------------- time series
def fill_gaps(x: np.ndarray, valid: np.ndarray) -> np.ndarray | None:
    """Linear interpolation over time of (T, ...) where `valid` is False; ends hold.
    Returns None if nothing is valid."""
    valid = np.asarray(valid, bool)
    if not valid.any():
        return None
    if valid.all():
        return x.copy()
    T = len(x)
    flat = x.reshape(T, -1).astype(np.float64)
    t = np.arange(T)
    out = np.stack([np.interp(t, t[valid], flat[valid, k]) for k in range(flat.shape[1])], 1)
    return out.reshape(x.shape)


def smooth_6d(d6: np.ndarray, window: int = 5, order: int = 2) -> np.ndarray:
    """Savitzky-Golay along time on 6D features, then re-orthonormalise."""
    T = d6.shape[0]
    w = min(window, T if T % 2 else T - 1)
    if w > order + 1:
        d6 = savgol_filter(d6, w, order, axis=0)
    return mat_to_6d(sixd_to_mat(d6))


# --------------------------------------------------------------------------- 2D keypoints
def hand_box(kp_px: np.ndarray, sc: np.ndarray, thr: float = 0.3, min_pts: int = 8):
    """Tight xyxy box around one hand's 21 keypoints, or None if too few are confident.
    WiLoR's own dataset pads it (rescale_factor), as it does for detector boxes."""
    ok = sc > thr
    if ok.sum() < min_pts:
        return None
    p = kp_px[ok]
    x1, y1 = p.min(0)
    x2, y2 = p.max(0)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    return [float(x1), float(y1), float(x2), float(y2)]


def segments_from_wrists(kp_px: np.ndarray, sc: np.ndarray, thr: float = 0.3,
                         min_len: int = 4, min_gap: int = 6) -> np.ndarray:
    """Rough per-frame sign segmentation in SignSparK's labels (0 non-sign, 2 start, 1 continuation).

    A frame is active when either wrist is raised above (shoulder line + 1.5 shoulder
    widths); an active run is split at pronounced speed minima (holds between signs).
    A stand-in for FAST: SignSparK only uses it to pick first/middle/last keyframes.
    """
    T = len(kp_px)
    lab = np.zeros(T, np.int32)
    sh_ok = (sc[:, COCO_L_SHOULDER] > thr) & (sc[:, COCO_R_SHOULDER] > thr)
    if not sh_ok.any():
        return lab
    s = np.median(np.linalg.norm(kp_px[sh_ok, COCO_L_SHOULDER] - kp_px[sh_ok, COCO_R_SHOULDER], axis=-1))
    y_sh = np.median((kp_px[sh_ok, COCO_L_SHOULDER, 1] + kp_px[sh_ok, COCO_R_SHOULDER, 1]) / 2)
    raised = np.zeros(T, bool)
    speed = np.zeros(T)
    for w in (COCO_L_WRIST, COCO_R_WRIST):
        ok = sc[:, w] > thr
        raised |= ok & (kp_px[:, w, 1] < y_sh + 1.5 * s)
        pos = fill_gaps(kp_px[:, w], ok)
        if pos is not None:
            v = np.r_[0.0, np.linalg.norm(np.diff(pos, axis=0), axis=-1) / max(s, 1e-6)]
            speed = np.maximum(speed, v)
    speed = np.convolve(speed, np.ones(3) / 3, mode='same')
    # close short gaps, drop short runs
    active = raised.copy()
    for run in _runs(~active):
        if run[0] > 0 and run[1] < T and run[1] - run[0] <= 3:
            active[run[0]:run[1]] = True
    for a, b in _runs(active):
        if b - a < min_len:
            active[a:b] = False
    for a, b in _runs(active):
        seg = speed[a:b]
        thr_v = 0.35 * np.median(seg) if len(seg) else 0.0
        cuts, last = [], a
        for t in range(a + 1, b - 1):
            if speed[t] < thr_v and speed[t] <= speed[t - 1] and speed[t] <= speed[t + 1] \
                    and t - last >= min_gap and b - t >= min_len:
                cuts.append(t)
                last = t
        # SignSparK's _segment_bounds only opens a segment on a 0 -> sign transition, so a
        # boundary inside a run must be a 0 frame (the hold) followed by a 2.
        lab[a:b] = 1
        lab[a] = 2
        for t in cuts:
            lab[t] = 0
            lab[t + 1] = 2
    return lab


def _runs(mask: np.ndarray):
    """[start, end) of True runs."""
    m = np.r_[False, np.asarray(mask, bool), False]
    d = np.flatnonzero(np.diff(m.astype(np.int8)))
    return list(zip(d[::2], d[1::2]))


# --------------------------------------------------------------------------- assembly
def assemble(nlf_pose_aa: np.ndarray, hands: dict) -> dict:
    """Combine NLF body and WiLoR hands into SMPL-X local rotations and SignSparK features.

    nlf_pose_aa: (T, 55, 3) NLF SMPL-X local axis-angle (joint 0 = global orient).
    hands[side] for side in ('left', 'right'): dict with
        'global_orient' (T, 3, 3), 'hand_pose' (T, 15, 3, 3) raw WiLoR output
        (left = mirrored-crop prediction), 'valid' (T,) bool.
    Returns features for the LMDB, the composed local rotations (T, 55, 3, 3) for QA,
    and per-side coverage.
    """
    T = nlf_pose_aa.shape[0]
    R = aa_to_mat(nlf_pose_aa)                               # (T, 55, 3, 3) local
    G = global_rotations(R, 22)
    out_local = R.copy()
    coverage = {}
    for side, elbow, wrist, hsl in (('left', L_ELBOW, L_WRIST, L_HAND), ('right', R_ELBOW, R_WRIST, R_HAND)):
        h = hands.get(side)
        valid = np.zeros(T, bool) if h is None else np.asarray(h['valid'], bool)
        coverage[side] = float(valid.mean())
        if not valid.any():
            continue                                         # keep NLF's own wrist and fingers
        go, hp = np.asarray(h['global_orient'], np.float64), np.asarray(h['hand_pose'], np.float64)
        if side == 'left':
            go = M_MIRROR @ go @ M_MIRROR                     # mirrored crop -> real camera
            hp = F_FLIP @ hp @ F_FLIP                         # raw (right-hand convention) -> SMPL-X left
        wrist_loc = np.swapaxes(G[:, elbow], -1, -2) @ go    # G_elbow^T @ R_hand_camera
        # gaps: interpolate in 6D between detected frames, then orthonormalise
        w6 = fill_gaps(mat_to_6d(wrist_loc), valid)
        h6 = fill_gaps(mat_to_6d(hp), valid)
        out_local[:, wrist] = sixd_to_mat(w6)
        out_local[:, hsl] = sixd_to_mat(h6)

    body6 = smooth_6d(mat_to_6d(out_local[:, 1:22]))
    left6 = smooth_6d(mat_to_6d(out_local[:, L_HAND]))
    right6 = smooth_6d(mat_to_6d(out_local[:, R_HAND]))
    jaw6 = smooth_6d(mat_to_6d(out_local[:, JAW]))
    # write the smoothed rotations back so QA sees exactly what goes into the LMDB
    out_local[:, 1:22] = sixd_to_mat(body6)
    out_local[:, L_HAND] = sixd_to_mat(left6)
    out_local[:, R_HAND] = sixd_to_mat(right6)
    out_local[:, JAW] = sixd_to_mat(jaw6)
    feats = {
        'left_features': left6.reshape(T, 90).astype(np.float32),
        'right_features': right6.reshape(T, 90).astype(np.float32),
        'body_features': body6.reshape(T, 126).astype(np.float32),
        'face_features': np.concatenate([jaw6.reshape(T, 6), np.zeros((T, 50))], 1).astype(np.float32),
    }
    return {'features': feats, 'local': out_local, 'coverage': coverage}


# --------------------------------------------------------------------------- LMDB
def record_bytes(language: str, text: str, segment: np.ndarray, feats: dict) -> bytes:
    """One clip in SignSparK's LMDB schema (DATA.md section 2)."""
    buf = io.BytesIO()
    np.savez(buf, language=np.array([language], dtype=object), translation=np.array([text], dtype=object),
             gloss=np.array([''], dtype=object), segment=np.asarray(segment, np.int32), **feats)
    return buf.getvalue()


def write_lmdb(path: str, records: list[tuple[str, bytes]], map_size: int = 1 << 34) -> None:
    import lmdb
    env = lmdb.open(path, map_size=map_size)
    with env.begin(write=True) as txn:
        for key, blob in records:
            txn.put(key.encode(), blob)
        txn.put(b'__meta__', pickle.dumps({'clip_ids': [k for k, _ in records], 'num_clips': len(records)}))
    env.close()


# --------------------------------------------------------------------------- MANO without chumpy
class _ChumpyStub:
    """Stands in for chumpy classes while unpickling MANO_*.pkl (chumpy does not install on
    current numpy/Python); a chumpy array keeps its values in state['x']."""

    def __init__(self, *args, **kwargs):
        pass

    def __setstate__(self, state):
        self.__dict__['_state'] = state


class _StubUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('chumpy'):
            return _ChumpyStub
        return super().find_class(module, name)


def mano_pkl_to_npz(pkl_path: str, npz_path: str) -> list[str]:
    """Convert MANO_RIGHT.pkl to an .npz smplx can load with ext='npz'. Returns the keys.

    The official MANO_RIGHT.pkl stores some arrays as chumpy *operations* (e.g. a
    Select over another array, with state {'a', 'idxs'} and no 'x'), which only real
    chumpy can evaluate. So real chumpy is used when importable (the current source
    at github.com/mattloper/chumpy imports fine on numpy 2 without installing); the
    stub only handles plain chumpy arrays.
    """
    try:
        import chumpy  # noqa: F401
        with open(pkl_path, 'rb') as fh:
            data = pickle.load(fh, encoding='latin1')
    except ImportError:
        with open(pkl_path, 'rb') as fh:
            data = _StubUnpickler(fh, encoding='latin1').load()
    out = {}
    for k, v in data.items():
        if isinstance(v, _ChumpyStub):
            if 'x' not in v._state:
                raise ValueError(f'{k}: chumpy operation, not a plain array; put the chumpy source on sys.path')
            v = v._state['x']
        elif hasattr(v, 'r') and type(v).__module__.startswith('chumpy'):
            v = v.r
        if hasattr(v, 'toarray'):                            # scipy.sparse (J_regressor)
            v = v.toarray()
        out[k] = np.asarray(v) if not isinstance(v, str) else np.array(v)
    np.savez(npz_path, **out)
    return sorted(out)

In [5]:
sys.path[:0] = ['/content', CHUMPY]
import numpy as np
import smplx_pilot as sp

mano_npz = f'{WILOR}/mano_data/MANO_RIGHT.npz'
if not os.path.exists(mano_npz):
    print('MANO keys:', sp.mano_pkl_to_npz(MANO_PKL, mano_npz))
cfg_path = f'{WILOR}/pretrained_models/model_config.yaml'
cfg_txt = open(cfg_path).read()
if 'EXT: npz' not in cfg_txt:
    cfg_txt = cfg_txt.replace('MANO:\n', 'MANO:\n  EXT: npz\n', 1)
    open(cfg_path, 'w').write(cfg_txt)
print('WiLoR MANO config ->', [l.strip() for l in cfg_txt.split('MANO:')[1].split('\n')[:8] if l.strip()])

MANO keys: ['J', 'J_regressor', 'bs_style', 'bs_type', 'f', 'hands_coeffs', 'hands_components', 'hands_mean', 'kintree_table', 'posedirs', 'shapedirs', 'v_template', 'weights']
WiLoR MANO config -> ['EXT: npz', 'DATA_DIR: mano_data', 'MODEL_PATH: ${MANO.DATA_DIR}', 'GENDER: neutral', 'NUM_HAND_JOINTS: 15', 'MEAN_PARAMS: ${MANO.DATA_DIR}/mano_mean_params.npz', 'CREATE_BODY_POSE: false']


## 3. Pick the pilot clips

Communication **train** clips not on the exclusion list, evenly spaced over the sorted uids so they span many episodes and signers. Val/test are left untouched.

In [6]:
N_PILOT = 200
rows = [json.loads(l) for l in open(MANIFEST)]
excluded = {l.strip() for l in open(EXCLUDE) if l.strip() and not l.startswith('#')}
pool = sorted((r for r in rows if r.get('dataset') == 'auslandaily' and r.get('subset') == 'communication'
               and r.get('split') == 'train' and r['uid'] not in excluded), key=lambda r: r['uid'])
PILOT = [pool[int(i)] for i in np.linspace(0, len(pool) - 1, N_PILOT)]
episodes = {r['uid'].rsplit('_', 1)[0] for r in PILOT}
print(f'{len(pool)} eligible train clips -> {len(PILOT)} pilot clips from {len(episodes)} episodes')
for r in PILOT[:5]:
    print(' ', r['uid'], '|', r['text'])

12347 eligible train clips -> 200 pilot clips from 90 episodes
  ad-communication-video_10_0 | hello .
  ad-communication-video_10_48 | hey sally is this bendy .
  ad-communication-video_11_110 | i know it sounds unusual but it will make the hot chocolate tastier .
  ad-communication-video_11_60 | yes it was fun trudging through the snow it made me feel really warm .
  ad-communication-video_12_17 | why are you going to the shop .


## 4. rtmlib keypoints for the pilot clips (from the Drive pose chunks)

In [7]:
RAW = '/content/pilot_raw'
os.makedirs(RAW, exist_ok=True)
want = {f"{r['uid']}.npz" for r in PILOT} - set(os.listdir(RAW))
for chunk in sorted(glob.glob(f'{WORK}/pose/chunk_*.tar')):
    if not want:
        break
    with tarfile.open(chunk) as tf:
        members = [m for m in tf.getmembers() if os.path.basename(m.name) in want]
        for m in members:
            m.name = os.path.basename(m.name)
        tf.extractall(RAW, members=members, filter='data')
        want -= {m.name for m in members}
if want:
    raise SystemExit(f'{len(want)} pilot clips have no rtmlib pose, e.g. {sorted(want)[:3]}')

def load_rtm(uid):
    with np.load(f'{RAW}/{uid}.npz', allow_pickle=False) as z:
        meta = json.loads(str(z['meta']))
        kp = z['keypoints'].astype(np.float64) * [meta['width'], meta['height']]
        return kp, z['scores'].astype(np.float64), meta
print('rtmlib poses ready:', len(PILOT))

rtmlib poses ready: 200


## 5. Load NLF and WiLoR

NLF gets the whole frame as the person box (the clips are already cropped to the signer) and an explicit pinhole camera (55° FOV), which section 8 reuses for reprojection. WiLoR is loaded from its repo directory because its config uses relative MANO paths.

In [8]:
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'     # WiLoR's Lightning checkpoint predates weights_only=True
import torch, torchvision, cv2
from torch.utils.data import default_collate

nlf = torch.jit.load(NLF_PATH).cuda().eval()
NLF_ESTIMATE = getattr(nlf, 'estimate_parametric_batched', None) or getattr(nlf, 'estimate_smpl_batched')

os.chdir(WILOR)
sys.path.insert(0, WILOR)
from wilor.models import load_wilor
from wilor.datasets.vitdet_dataset import ViTDetDataset
wilor, wilor_cfg = load_wilor(checkpoint_path='./pretrained_models/wilor_final.ckpt', cfg_path='./pretrained_models/model_config.yaml')
wilor = wilor.cuda().eval()
os.chdir('/content')

FOV = 55.0
def intrinsics(W, H):
    f = max(W, H) / (2 * np.tan(np.radians(FOV) / 2))
    return np.array([[f, 0, W / 2], [0, f, H / 2], [0, 0, 1]], np.float64)
print('models loaded')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading  ./pretrained_models/wilor_final.ckpt


/usr/local/lib/python3.13/dist-packages/lightning_fabric/utilities/cloud_io.py:73: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint pretrained_models/wilor_final.ckpt`


models loaded


## 6. Fit every pilot clip (resumable)

Per clip: read the video from `Signer.zip` on Drive, NLF on all frames, WiLoR on every confident rtmlib hand box, save the raw fits to `params/<uid>.npz`. Clips already saved are skipped.

In [9]:
def read_video(zf, member):
    tmp = '/content/_clip.mp4'
    with open(tmp, 'wb') as fh:
        fh.write(zf.read(member))
    cap = cv2.VideoCapture(tmp)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr)
    cap.release()
    return frames, fps

@torch.inference_mode()
def run_nlf(frames_bgr, K):
    H, W = frames_bgr[0].shape[:2]
    imgs = torch.from_numpy(np.stack([f[:, :, ::-1] for f in frames_bgr])).permute(0, 3, 1, 2).contiguous().cuda()
    boxes = [torch.tensor([[0.0, 0.0, W, H, 1.0]], device='cuda')] * len(frames_bgr)
    Kt = torch.from_numpy(K).float()[None].cuda()
    with torch.device('cuda'):
        pred = NLF_ESTIMATE(imgs, boxes, intrinsic_matrix=Kt, model_name='smplx')
    get = lambda k: torch.stack([p[0] for p in pred[k]]).float().cpu().numpy()
    return {'nlf_pose': get('pose').reshape(len(frames_bgr), 55, 3), 'nlf_betas': get('betas'),
            'nlf_trans': get('trans'), 'nlf_joints2d': get('joints2d')}

@torch.inference_mode()
def run_wilor(frames_bgr, kp, sc, batch=64):
    T = len(frames_bgr)
    out = {s: {'global_orient': np.tile(np.eye(3), (T, 1, 1)), 'hand_pose': np.tile(np.eye(3), (T, 15, 1, 1)),
               'valid': np.zeros(T, bool)} for s in ('left', 'right')}
    items, where = [], []
    for t in range(T):
        boxes, right = [], []
        for side, sl in (('left', sp.COCO_L_HAND), ('right', sp.COCO_R_HAND)):
            b = sp.hand_box(kp[t, sl], sc[t, sl])
            if b is not None:
                boxes.append(b); right.append(1.0 if side == 'right' else 0.0); where.append((t, side))
        if boxes:
            ds = ViTDetDataset(wilor_cfg, frames_bgr[t], np.array(boxes), np.array(right), rescale_factor=2.0)
            items += [ds[i] for i in range(len(ds))]
    for i in range(0, len(items), batch):
        b = default_collate(items[i:i + batch])
        b = {k: v.cuda() if torch.is_tensor(v) else v for k, v in b.items()}
        pm = wilor(b)['pred_mano_params']
        go = pm['global_orient'].reshape(-1, 3, 3).float().cpu().numpy()
        hp = pm['hand_pose'].reshape(-1, 15, 3, 3).float().cpu().numpy()
        for j, (t, side) in enumerate(where[i:i + batch]):
            out[side]['global_orient'][t] = go[j]; out[side]['hand_pose'][t] = hp[j]; out[side]['valid'][t] = True
    return out

zf = zipfile.ZipFile(SIGNER_ZIP)
t_start, done = time.time(), 0
for n, r in enumerate(PILOT):
    dst = f"{OUT}/params/{r['uid']}.npz"
    if os.path.exists(dst):
        continue
    frames, fps = read_video(zf, r['video'].split('::', 1)[1])
    kp, sc, meta = load_rtm(r['uid'])
    H, W = frames[0].shape[:2]
    if (W, H) != (meta['width'], meta['height']):
        raise RuntimeError(f"{r['uid']}: video {W}x{H} but rtmlib meta {meta['width']}x{meta['height']}")
    T = min(len(frames), len(kp))
    if abs(len(frames) - len(kp)) > 1:
        print(f"  warning {r['uid']}: {len(frames)} video frames vs {len(kp)} pose frames; using {T}")
    frames, kp, sc = frames[:T], kp[:T], sc[:T]
    K = intrinsics(W, H)
    body = run_nlf(frames, K)
    hands = run_wilor(frames, kp, sc)
    save = dict(body, K=K, W=W, H=H, fps=fps, T=T)
    for s in ('left', 'right'):
        for k, v in hands[s].items():
            save[f'{s}_{k}'] = v
    tmp = dst + '.tmp.npz'
    np.savez_compressed(tmp, **save)
    os.replace(tmp, dst)
    done += 1
    if done % 10 == 0 or n == len(PILOT) - 1:
        el = time.time() - t_start
        print(f'{n + 1}/{len(PILOT)} clips | {done} fitted this session | {el / done:.1f} s/clip')
print('all pilot clips fitted:', len(glob.glob(f'{OUT}/params/*.npz')))

/content/WiLoR/wilor/utils/geometry.py:61: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  b3 = torch.cross(b1, b2)


10/200 clips | 10 fitted this session | 14.5 s/clip
20/200 clips | 20 fitted this session | 10.0 s/clip
30/200 clips | 30 fitted this session | 8.6 s/clip
40/200 clips | 40 fitted this session | 8.4 s/clip
50/200 clips | 50 fitted this session | 8.2 s/clip
60/200 clips | 60 fitted this session | 8.1 s/clip
70/200 clips | 70 fitted this session | 7.6 s/clip
80/200 clips | 80 fitted this session | 7.3 s/clip
90/200 clips | 90 fitted this session | 7.0 s/clip
100/200 clips | 100 fitted this session | 6.7 s/clip
110/200 clips | 110 fitted this session | 6.6 s/clip
120/200 clips | 120 fitted this session | 6.6 s/clip
130/200 clips | 130 fitted this session | 6.4 s/clip
140/200 clips | 140 fitted this session | 6.2 s/clip
150/200 clips | 150 fitted this session | 6.1 s/clip
160/200 clips | 160 fitted this session | 5.9 s/clip
170/200 clips | 170 fitted this session | 5.8 s/clip
180/200 clips | 180 fitted this session | 5.8 s/clip
190/200 clips | 190 fitted this session | 5.8 s/clip
200/200 c

## 7. Assemble, segment and write the LMDB

Fast and deterministic from the saved fits, so it can be re-run after any change to `smplx_pilot.py`.

In [10]:
def load_params(uid):
    with np.load(f'{OUT}/params/{uid}.npz') as z:
        p = {k: z[k] for k in z.files}
    hands = {s: {'global_orient': p[f'{s}_global_orient'], 'hand_pose': p[f'{s}_hand_pose'], 'valid': p[f'{s}_valid']}
             for s in ('left', 'right')}
    return p, hands

records, stats = [], []
ASM = {}
for r in PILOT:
    p, hands = load_params(r['uid'])
    kp, sc, _ = load_rtm(r['uid'])
    T = int(p['T'])
    res = sp.assemble(p['nlf_pose'], hands)
    seg = sp.segments_from_wrists(kp[:T], sc[:T])
    records.append((r['uid'], sp.record_bytes('Auslan', r['text'], seg, res['features'])))
    ASM[r['uid']] = res
    n_seg = int((seg == 2).sum())
    stats.append({'uid': r['uid'], 'T': T, 'fps': float(p['fps']), 'words': len(r['text'].split()),
                  'left_cov': res['coverage']['left'], 'right_cov': res['coverage']['right'],
                  'active': float((seg > 0).mean()), 'segments': n_seg})

LMDB_LOCAL = '/content/pilot_lmdb/train/AuslanDaily-pilot_train.lmdb'
shutil.rmtree(os.path.dirname(LMDB_LOCAL), ignore_errors=True)
os.makedirs(os.path.dirname(LMDB_LOCAL))
sp.write_lmdb(LMDB_LOCAL, records)
shutil.copytree(LMDB_LOCAL, f'{OUT}/lmdb/train/AuslanDaily-pilot_train.lmdb', dirs_exist_ok=True)

import pandas as pd
st = pd.DataFrame(stats)
st.to_csv(f'{OUT}/clip_stats.csv', index=False)
print(f'{len(records)} records -> {OUT}/lmdb/train')
print(st[['T', 'fps', 'words', 'left_cov', 'right_cov', 'active', 'segments']].describe().round(2).to_string())
print('\nclips with a hand seen in < 50% of frames:', int(((st.left_cov < 0.5) & (st.right_cov < 0.5)).sum()))

200 records -> /content/drive/MyDrive/auslan_work/smplx_pilot/lmdb/train
            T    fps   words  left_cov  right_cov  active  segments
count  200.00  200.0  200.00    200.00     200.00  200.00    200.00
mean    77.17   25.0    8.31      0.98       0.99    0.97      2.54
std     52.31    0.0    5.65      0.10       0.10    0.12      1.81
min     10.00   25.0    2.00      0.03       0.03    0.00      0.00
25%     42.75   25.0    4.00      1.00       1.00    1.00      1.00
50%     62.50   25.0    7.00      1.00       1.00    1.00      2.00
75%     96.00   25.0   11.00      1.00       1.00    1.00      3.00
max    357.00   25.0   39.00      1.00       1.00    1.00     12.00

clips with a hand seen in < 50% of frames: 2


## 8. Quality: reprojection against rtmlib

- **NLF self-check**: our SMPL-X forward pass of NLF's own parameters, projected with our camera, must land on NLF's `joints2d`. If it does not, the body-model or translation conventions differ; the residual is reported and removed by a per-frame 2D shift before the other checks.
- **Arms**: shoulders, elbows, wrists vs rtmlib, error in shoulder widths.
- **Hands**: 21 points per hand, wrist-aligned, error in units of the rtmlib hand size (wrist to middle-finger base), and PCK@0.2. **Assembled (WiLoR) vs NLF-only hands**: the assembled hands must be clearly better; if they are worse, a convention is wrong.

Only frames where the rtmlib point is confident (score > 0.5) count.

In [11]:
import smplx
SMPLX_DIR = os.path.dirname(os.path.dirname(SMPLX_NPZ))
_bm = {}
def body_model(n_betas):
    if n_betas not in _bm:
        _bm[n_betas] = smplx.create(SMPLX_DIR, model_type='smplx', gender='neutral', use_pca=False, flat_hand_mean=True,
                                   num_betas=n_betas, num_expression_coeffs=10).cuda().eval()
    return _bm[n_betas]

@torch.no_grad()
def fk(local, betas, trans):
    aa = torch.from_numpy(sp.mat_to_aa(local).reshape(len(local), -1)).float().cuda()
    bm = body_model(betas.shape[1])
    out = bm(global_orient=aa[:, :3], body_pose=aa[:, 3:66], jaw_pose=aa[:, 66:69], leye_pose=aa[:, 69:72],
             reye_pose=aa[:, 72:75], left_hand_pose=aa[:, 75:120], right_hand_pose=aa[:, 120:165],
             betas=torch.from_numpy(betas).float().cuda(), transl=torch.from_numpy(trans).float().cuda(),
             expression=torch.zeros(len(local), 10, device='cuda'))
    return out.joints.cpu().numpy()                     # (T, 127, 3), metres, camera frame

def project(J, K):
    return (J[..., :2] / np.maximum(J[..., 2:], 1e-3)) @ K[:2, :2].T + K[:2, 2]

def hand_err(pred2d, kp, sc, side):
    sl = sp.COCO_L_HAND if side == 'left' else sp.COCO_R_HAND
    g, s = kp[:, sl], sc[:, sl]
    p = pred2d[:, sp.OP_FROM_SMPLX[side]]
    ok = (s > 0.5) & (s[:, :1] > 0.5)                  # need a confident wrist to align
    size = np.linalg.norm(g[:, 9] - g[:, 0], axis=-1)[:, None]
    e = np.linalg.norm((p - p[:, :1]) - (g - g[:, :1]), axis=-1) / np.maximum(size, 1e-6)
    return e[ok & (size > 3)]

rows_qa = []
for r in PILOT:
    p, _ = load_params(r['uid'])
    kp, sc, _ = load_rtm(r['uid'])
    T, K = int(p['T']), p['K']
    kp, sc = kp[:T], sc[:T]
    nlf_local = sp.aa_to_mat(p['nlf_pose'])
    j_nlf = project(fk(nlf_local, p['nlf_betas'], p['nlf_trans']), K)
    ref = p['nlf_joints2d'][:, :22]
    shift = (ref - j_nlf[:, :22]).mean(1, keepdims=True)                # NLF-vs-smplx convention residual
    self_err = np.linalg.norm(ref - j_nlf[:, :22], axis=-1).mean()
    j_nlf = j_nlf + shift
    j_asm = project(fk(ASM[r['uid']]['local'], p['nlf_betas'], p['nlf_trans']), K) + shift
    sw = np.median(np.linalg.norm(kp[:, 5] - kp[:, 6], axis=-1))
    arm = [np.linalg.norm(j_asm[:, sj] - kp[:, cj], axis=-1)[sc[:, cj] > 0.5] / sw for cj, sj in sp.COCO_TO_SMPLX_ARMS.items()]
    row = {'uid': r['uid'], 'self_px': self_err, 'arm_err': float(np.mean(np.concatenate(arm)))}
    for side in ('left', 'right'):
        ea, en = hand_err(j_asm, kp, sc, side), hand_err(j_nlf, kp, sc, side)
        row[f'{side}_asm'] = float(ea.mean()) if len(ea) else np.nan
        row[f'{side}_nlf'] = float(en.mean()) if len(en) else np.nan
        row[f'{side}_pck_asm'] = float((ea < 0.2).mean()) if len(ea) else np.nan
        row[f'{side}_pck_nlf'] = float((en < 0.2).mean()) if len(en) else np.nan
    rows_qa.append(row)
qa = pd.DataFrame(rows_qa)
qa.to_csv(f'{OUT}/qa.csv', index=False)
print(f"NLF self-check: mean {qa.self_px.mean():.1f} px (max clip {qa.self_px.max():.1f}) before the 2D shift")
print(f"arms (shoulder/elbow/wrist): {qa.arm_err.mean():.3f} shoulder widths")
print(f"{'':<7}{'hand err asm':>14}{'hand err NLF':>14}{'PCK@0.2 asm':>13}{'PCK@0.2 NLF':>13}")
for side in ('left', 'right'):
    print(f"{side:<7}{qa[f'{side}_asm'].mean():>14.3f}{qa[f'{side}_nlf'].mean():>14.3f}"
          f"{qa[f'{side}_pck_asm'].mean():>13.3f}{qa[f'{side}_pck_nlf'].mean():>13.3f}")
worst = qa.assign(h=qa[['left_asm', 'right_asm']].mean(1)).sort_values('h', ascending=False)
print('\nworst clips by hand error:'); print(worst[['uid', 'left_asm', 'right_asm', 'arm_err']].head(8).round(3).to_string(index=False))

NLF self-check: mean 3.6 px (max clip 718.8) before the 2D shift
arms (shoulder/elbow/wrist): 0.130 shoulder widths
         hand err asm  hand err NLF  PCK@0.2 asm  PCK@0.2 NLF
left            0.860         0.795        0.104        0.114
right           0.831         0.874        0.116        0.099

worst clips by hand error:
                          uid  left_asm  right_asm  arm_err
  ad-communication-video_69_2     0.779      6.956    0.531
   ad-communication-video_7_2     1.640      4.793    0.350
  ad-communication-video_9_99     1.532      3.547    0.202
ad-communication-video_82_180     3.960      0.857    0.218
 ad-communication-video_35_11     1.931      1.749    0.105
ad-communication-video_67_230     2.413      1.117    0.122
 ad-communication-video_49_73     0.869      2.556    0.139
ad-communication-video_63_149     1.997      1.294    0.129


## 9. Overlay videos

Video frames with rtmlib keypoints (green) and the assembled SMPL-X joints projected back (red: arms and hands). Shows the 3 best and the 3 worst clips by hand error, so you see both ends.

In [12]:
from IPython.display import Video, display
HAND_BONES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (0, 9), (9, 10), (10, 11), (11, 12),
              (0, 13), (13, 14), (14, 15), (15, 16), (0, 17), (17, 18), (18, 19), (19, 20)]
ARM_BONES = [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10)]

def draw_skel(img, pts_body, pts_l, pts_r, colour):
    for a, b in ARM_BONES:
        cv2.line(img, tuple(map(int, pts_body[a])), tuple(map(int, pts_body[b])), colour, 2, cv2.LINE_AA)
    for hp in (pts_l, pts_r):
        for a, b in HAND_BONES:
            cv2.line(img, tuple(map(int, hp[a])), tuple(map(int, hp[b])), colour, 1, cv2.LINE_AA)

def overlay(uid, path):
    r = next(x for x in PILOT if x['uid'] == uid)
    frames, fps = read_video(zf, r['video'].split('::', 1)[1])
    p, _ = load_params(uid)
    kp, sc, _ = load_rtm(uid)
    T, K = int(p['T']), p['K']
    j_nlf = project(fk(sp.aa_to_mat(p['nlf_pose']), p['nlf_betas'], p['nlf_trans']), K)
    shift = (p['nlf_joints2d'][:, :22] - j_nlf[:, :22]).mean(1, keepdims=True)
    j = project(fk(ASM[uid]['local'], p['nlf_betas'], p['nlf_trans']), K) + shift
    body = np.zeros((T, 11, 2)); body[:, 5:11] = j[:, [16, 17, 18, 19, 20, 21]]
    tmp = '/content/_ov.mp4'
    H, W = frames[0].shape[:2]
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps or 25, (W, H))
    for t in range(T):
        img = frames[t].copy()
        draw_skel(img, kp[t], kp[t, sp.COCO_L_HAND], kp[t, sp.COCO_R_HAND], (0, 200, 0))
        draw_skel(img, body[t], j[t, sp.OP_FROM_SMPLX['left']], j[t, sp.OP_FROM_SMPLX['right']], (0, 0, 230))
        vw.write(img)
    vw.release()
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', path], check=True)

order = qa.assign(h=qa[['left_asm', 'right_asm']].mean(1)).dropna(subset=['h']).sort_values('h')
pick = list(order.uid[:3]) + list(order.uid[-3:])
for uid in pick:
    path = f'{OUT}/videos/{uid}.mp4'
    overlay(uid, path)
    row = order[order.uid == uid].iloc[0]
    text = next(x['text'] for x in PILOT if x['uid'] == uid)
    print(f"{uid} | hand err L {row.left_asm:.3f} R {row.right_asm:.3f} | {text}")
    display(Video(path, embed=True, width=480))

ad-communication-video_87_152 | hand err L 0.341 R 0.273 | another one .


ad-communication-video_23_126 | hand err L 0.355 R 0.385 | just teasing .


ad-communication-video_36_39 | hand err L 0.343 R 0.424 | what can we eat with .


ad-communication-video_9_99 | hand err L 1.532 R 3.547 | bye .


ad-communication-video_7_2 | hand err L 1.640 R 4.793 | look .


ad-communication-video_69_2 | hand err L 0.779 R 6.956 | look .


## 10. SignSparK reads the LMDB

In [13]:
res = subprocess.run([sys.executable, '-c', f'''
import sys, types, numpy as np
sys.path.insert(0, "{SSK}"); sys.modules["wandb"] = types.ModuleType("wandb")
from types import SimpleNamespace
from signspark.pose_datasets_lmdb import PoseDataset_Lmdb
for feat, concat, dim in (("hand", True, 180), ("body", False, 60), ("face", False, 56)):
    a = SimpleNamespace(dataset_feat=feat, specify_lang=True, flip_left_hand=True, keyframe_selection_mode=3, custom_keyframe_file="null")
    ds = PoseDataset_Lmdb(a, split="train", seq_len=304, data="{LMDB_LOCAL}", concat_hands=concat)
    bad = n_kf = 0
    for i in range(len(ds)):
        x, L, text, kfs, name = ds[i]
        bad += (not np.isfinite(x.numpy()).all()) or x.shape != (304, dim)
        n_kf += len(kfs)
    ds.env.close(); ds._env = None      # the next stream re-opens the same LMDB in this process
    print(feat, len(ds), "clips | bad", bad, "| keyframes/clip", round(n_kf / len(ds), 1), "| e.g.", repr(text))
'''], capture_output=True, text=True)
print(res.stdout[-3000:], res.stderr[-3000:])

Logging to /tmp/openai-2026-09-21-10-15-56-291646
### flip_left_hand=True: converting left hand from SMPLX-left to right-hand convention
############### 
Loading [Dataset: hand @ 200 items | Seq_Len: 304] from /content/pilot_lmdb/train/AuslanDaily-pilot_train.lmdb
hand 200 clips | bad 0 | keyframes/clip 3.0 | e.g. '<Auslan> bye .'
### flip_left_hand=True: converting left hand from SMPLX-left to right-hand convention
############### 
Loading [Dataset: body @ 200 items | Seq_Len: 304] from /content/pilot_lmdb/train/AuslanDaily-pilot_train.lmdb
body 200 clips | bad 0 | keyframes/clip 3.0 | e.g. '<Auslan> bye .'
### flip_left_hand=True: converting left hand from SMPLX-left to right-hand convention
############### 
Loading [Dataset: face @ 200 items | Seq_Len: 304] from /content/pilot_lmdb/train/AuslanDaily-pilot_train.lmdb
face 200 clips | bad 0 | keyframes/clip 3.0 | e.g. '<Auslan> bye .'
 


## 11. Diagnostic: WiLoR's own 2D hands vs assembled hands vs rtmlib

Section 8's hand PCK is low for both the assembled and the NLF-only hands, while the overlays mostly look aligned; only the two `look .` clips had exaggerated hands. This separates the possible causes on 15 clips (5 best, 5 median, 5 worst by section 8's hand error), using medians:

- `hand px`: rtmlib hand size (wrist to middle-finger base) in pixels. If it is small (under ~15 px), rtmlib's own hand points are coarse and the PCK threshold is harsher than it looks.
- `WiLoR err / PCK`: WiLoR's own prediction, projected with WiLoR's camera exactly as its demo does, wrist-aligned, vs rtmlib. `WiLoR abs`: without wrist alignment.
- `asm err / PCK`: the assembled SMPL-X hands (section 8's measure, as medians).

Reading: WiLoR good but assembled worse → the assembly (wrist composition or a convention) is wrong. Both similar and poor → not the assembly; the reference or WiLoR's accuracy at this resolution. Only `worst` off → per-frame WiLoR failures (the `look .` clips), fixed by rejecting frames where WiLoR disagrees with rtmlib.

Needs sections 1–5 and 7–8 run in this session (section 6 skips fitted clips).

In [ ]:
from wilor.utils.renderer import cam_crop_to_full

@torch.no_grad()
def wilor_2d(frames, kp, sc):
    T = len(frames)
    out = {s: np.full((T, 21, 2), np.nan) for s in ('left', 'right')}
    items, where = [], []
    for t in range(T):
        boxes, right = [], []
        for side, sl in (('left', sp.COCO_L_HAND), ('right', sp.COCO_R_HAND)):
            b = sp.hand_box(kp[t, sl], sc[t, sl])
            if b is not None:
                boxes.append(b); right.append(1.0 if side == 'right' else 0.0); where.append((t, side))
        if boxes:
            ds = ViTDetDataset(wilor_cfg, frames[t], np.array(boxes), np.array(right), rescale_factor=2.0)
            items += [ds[i] for i in range(len(ds))]
    for i in range(0, len(items), 64):
        b = default_collate(items[i:i + 64])
        b = {k: v.cuda() if torch.is_tensor(v) else v for k, v in b.items()}
        o = wilor(b)
        mult = 2 * b['right'].float() - 1                       # left hands were predicted on the mirrored crop
        cam = o['pred_cam'].clone(); cam[:, 1] = mult * cam[:, 1]
        img_size = b['img_size'].float()
        focal = wilor_cfg.EXTRA.FOCAL_LENGTH / wilor_cfg.MODEL.IMAGE_SIZE * img_size.max()
        cam_t = cam_crop_to_full(cam, b['box_center'].float(), b['box_size'].float(), img_size, focal)
        j3 = o['pred_keypoints_3d'].clone(); j3[..., 0] = mult[:, None] * j3[..., 0]
        pts = j3 + cam_t[:, None, :]
        uv = (pts[..., :2] / pts[..., 2:] * focal + img_size[:, None, :] / 2).cpu().numpy()
        for j, (t, side) in enumerate(where[i:i + 64]):
            out[side][t] = uv[j]
    return out

def err21(pred, kp, sc, side):
    sl = sp.COCO_L_HAND if side == 'left' else sp.COCO_R_HAND
    g, s = kp[:, sl], sc[:, sl]
    size = np.linalg.norm(g[:, 9] - g[:, 0], axis=-1)
    ok = (s > 0.5) & (s[:, :1] > 0.5) & np.isfinite(pred[..., 0]) & (size[:, None] > 3)
    e_al = np.linalg.norm((pred - pred[:, :1]) - (g - g[:, :1]), axis=-1) / np.maximum(size[:, None], 1e-6)
    e_abs = np.linalg.norm(pred - g, axis=-1) / np.maximum(size[:, None], 1e-6)
    return e_al[ok], e_abs[ok], size[(s[:, 0] > 0.5) & (s[:, 9] > 0.5)]

o = qa.assign(h=qa[['left_asm', 'right_asm']].mean(1)).dropna(subset=['h']).sort_values('h')
mid = len(o) // 2
groups = {'best': list(o.uid[:5]), 'median': list(o.uid[mid - 2: mid + 3]), 'worst': list(o.uid[-5:])}
print(f"{'group':<7}{'W x H':>11}{'hand px':>9}{'WiLoR err':>11}{'WiLoR PCK':>11}{'WiLoR abs':>11}{'asm err':>9}{'asm PCK':>9}")
DIAG = {}
for g_name, uids in groups.items():
    acc = {k: [] for k in ('w_al', 'w_abs', 'a_al', 'size')}
    sizes_wh = set()
    for uid in uids:
        r = next(x for x in PILOT if x['uid'] == uid)
        frames, _ = read_video(zf, r['video'].split('::', 1)[1])
        p, _ = load_params(uid)
        kp, sc, meta = load_rtm(uid)
        T, K = int(p['T']), p['K']
        frames, kp, sc = frames[:T], kp[:T], sc[:T]
        sizes_wh.add(f"{meta['width']}x{meta['height']}")
        w2 = wilor_2d(frames, kp, sc)
        j_nlf = project(fk(sp.aa_to_mat(p['nlf_pose']), p['nlf_betas'], p['nlf_trans']), K)
        shift = (p['nlf_joints2d'][:, :22] - j_nlf[:, :22]).mean(1, keepdims=True)
        j_asm = project(fk(ASM[uid]['local'], p['nlf_betas'], p['nlf_trans']), K) + shift
        DIAG[uid] = {'wilor2d': w2, 'asm2d': j_asm}
        for side in ('left', 'right'):
            a_, b_, s_ = err21(w2[side], kp, sc, side)
            acc['w_al'].append(a_); acc['w_abs'].append(b_); acc['size'].append(s_)
            acc['a_al'].append(err21(j_asm[:, sp.OP_FROM_SMPLX[side]], kp, sc, side)[0])
    c = {k: np.concatenate(v) if any(len(x) for x in v) else np.array([np.nan]) for k, v in acc.items()}
    wh = '/'.join(sorted(sizes_wh)) if len(sizes_wh) <= 2 else 'mixed'
    print(f"{g_name:<7}{wh:>11}{np.median(c['size']):>9.1f}"
          f"{np.median(c['w_al']):>11.3f}{(c['w_al'] < 0.2).mean():>11.3f}{np.median(c['w_abs']):>11.3f}"
          f"{np.median(c['a_al']):>9.3f}{(c['a_al'] < 0.2).mean():>9.3f}")

## What to send back

1. Section 7's table (hand coverage, segments per clip) and the count of clips with poor hand coverage.
2. Section 8's summary: NLF self-check px, arm error, and the hand table (**assembled vs NLF**).
3. Your impression of the section 9 videos: do the red hands follow the green ones, including palm direction and finger shapes? Is left/right ever swapped?
4. Section 10's keyframes per clip (should now exceed 3: sign boundaries are written as a 0 frame before each new sign) and section 11's table.